EasyWave Model (CPU):

In [11]:
import subprocess
import glob
import os
import shutil
from pathlib import Path
import numpy as np
import xarray as xr
import struct

# ================== FUNCIONES AUXILIARES ==================
def leer_grilla_easywave_ascii(grd_file):
    with open(grd_file, 'r') as f:
        assert f.readline().strip() == 'DSAA'
        ncols, nrows = map(int, f.readline().split())
        xmin, xmax = map(float, f.readline().split())
        ymin, ymax = map(float, f.readline().split())
        _ = f.readline()
        z = np.empty((nrows, ncols), dtype=np.float32)
        for r in range(nrows):
            vals = f.readline().split()
            row = np.zeros(ncols, dtype=np.float32)
            for c in range(min(len(vals), ncols)):
                row[c] = float(vals[c])
            z[r, :] = row
    return z, nrows, ncols, xmin, xmax, ymin, ymax

def leer_sshmax_subdom(path):
    path = Path(path)
    with path.open('rb') as f:
        if f.read(4) != b'DSBB':
            raise ValueError(f'{path} no comienza con DSBB')
        hdr = f.read(52)
        nI, nJ, loMin, loMax, laMin, laMax, t0, t1 = struct.unpack('<hh6d', hdr)
        _ = struct.unpack('<2d', f.read(16))
        data = np.fromfile(f, dtype='<f4')
        if data.size < nI*nJ:
            tmp = np.zeros(nI*nJ, dtype=np.float32)
            tmp[:data.size] = data
            data = tmp
        arr = data.reshape((nJ, nI))
    return arr, (nI, nJ), (loMin, loMax, laMin, laMax)

def colocar_subdom_en_full(full_shape, domain_bounds, sub_arr, sub_bounds):
    nrows, ncols = full_shape
    xmin, xmax, ymin, ymax = domain_bounds
    loMin, loMax, laMin, laMax = sub_bounds

    lon_full = np.linspace(xmin, xmax, ncols)
    lat_full = np.linspace(ymin, ymax, nrows)

    i0 = int(np.argmin(np.abs(lon_full - loMin)))
    i1 = int(np.argmin(np.abs(lon_full - loMax))) + 1
    j0 = int(np.argmin(np.abs(lat_full - laMin)))
    j1 = int(np.argmin(np.abs(lat_full - laMax))) + 1

    nJ, nI = sub_arr.shape
    if (j1 - j0) != nJ:
        j1 = j0 + nJ
    if (i1 - i0) != nI:
        i1 = i0 + nI

    full = np.zeros((nrows, ncols), dtype=np.float32)
    full[j0:j1, i0:i1] = sub_arr
    return full, lat_full, lon_full

def generar_netcdf(grd_file, sshmax_file, nc_file):
    bathy, nrows, ncols, xmin, xmax, ymin, ymax = leer_grilla_easywave_ascii(grd_file)
    sub_arr, (nI, nJ), sub_bounds = leer_sshmax_subdom(sshmax_file)
    max_height_full, lat, lon = colocar_subdom_en_full(
        (nrows, ncols),
        (xmin, xmax, ymin, ymax),
        sub_arr,
        sub_bounds
    )
    ds = xr.Dataset(
        data_vars={
            "original_bathy": (["lat", "lon"], bathy),
            "deformed_bathy": (["lat", "lon"], bathy),
            "max_height":     (["lat", "lon"], max_height_full),
        },
        coords={"lat": lat, "lon": lon},
        attrs={"source": "EasyWave sshmax embebido en grilla completa"}
    )
    ds.to_netcdf(nc_file)
    print(f"✅ NetCDF listo: {nc_file}")

# ================== MAIN ==================
grd_file = r"Datos/EasyWave/GrdASCII/grilla_ascii.grd"
flt_base = r"Datos/simulaciones-tsunami-hysea"
out_folder = r"Datos/EasyWave/Outputs"
os.makedirs(out_folder, exist_ok=True)

flt_files = glob.glob(os.path.join(flt_base, "**", "*.flt"), recursive=True)

for flt_file in flt_files:
    rel_path = os.path.relpath(flt_file, flt_base)
    sim_name = os.path.splitext(rel_path.replace(os.sep, "_"))[0]

    run_out = os.path.join(out_folder, sim_name)
    os.makedirs(run_out, exist_ok=True)

    print(f"Procesando {sim_name} ...")

    subprocess.run([
        "easywave",
        "-grid", os.path.abspath(grd_file),
        "-source", os.path.abspath(flt_file),
        "-time", "120"
    ], cwd=run_out)

    subprocess.run(["sshmax2png.sh", "-grd", os.path.abspath(grd_file)], cwd=run_out)
    subprocess.run(["ssh2png.sh", "-grd", os.path.abspath(grd_file), "03600"], cwd=run_out)


    flt_dir = os.path.dirname(flt_file)
    file_heig = os.path.join(run_out, "eWave.2D.png")
    file_prop = os.path.join(run_out, "eWave.2D.03600.png")
    new_heig = os.path.join(flt_dir, f"max_wave_heights.png")
    new_prop = os.path.join(flt_dir, f"wave_propagation.png")

    if os.path.exists(file_heig):
        shutil.move(file_heig, new_heig)
    if os.path.exists(file_prop):
        shutil.move(file_prop, new_prop)

    # generar NetCDF 
    sshmax_file = os.path.join(run_out, "eWave.2D.sshmax")
    nc_file = os.path.join(flt_dir, f"resultado_easywave.nc")
    if os.path.exists(sshmax_file):
        generar_netcdf(grd_file, sshmax_file, nc_file)

print("✅ Todos los procesos han finalizado.")


Procesando mw_8.1_02083_02083 ...

easyWave ver.2013-04-11
Model time = 00:00:00,   elapsed: 228787 msec
Model time = 00:10:00,   elapsed: 229162 msec
Model time = 00:20:00,   elapsed: 230046 msec
Model time = 00:30:00,   elapsed: 231855 msec
Model time = 00:40:00,   elapsed: 234841 msec
Model time = 00:50:00,   elapsed: 238960 msec
Model time = 01:00:00,   elapsed: 243947 msec
Model time = 01:10:00,   elapsed: 249504 msec
Model time = 01:20:00,   elapsed: 255719 msec
Model time = 01:30:00,   elapsed: 262746 msec
Model time = 01:40:00,   elapsed: 270556 msec
Model time = 01:50:00,   elapsed: 279036 msec
Model time = 02:00:00,   elapsed: 288245 msec
✅ NetCDF listo: Datos/simulaciones-tsunami-hysea/mw_8.1/02083/resultado_easywave.nc
Procesando mw_8.3_03529_03529 ...

easyWave ver.2013-04-11
Model time = 00:00:00,   elapsed: 229298 msec
Model time = 00:10:00,   elapsed: 229764 msec
Model time = 00:20:00,   elapsed: 230839 msec
Model time = 00:30:00,   elapsed: 232850 msec
Model time = 00:

EasyWave Model (GPU):

In [ ]:
import subprocess
import glob
import os
import shutil
from pathlib import Path
import numpy as np
import xarray as xr
import struct

# ================== FUNCIONES AUXILIARES ==================
def leer_grilla_easywave_ascii(grd_file):
    with open(grd_file, 'r') as f:
        assert f.readline().strip() == 'DSAA'
        ncols, nrows = map(int, f.readline().split())
        xmin, xmax = map(float, f.readline().split())
        ymin, ymax = map(float, f.readline().split())
        _ = f.readline()
        z = np.empty((nrows, ncols), dtype=np.float32)
        for r in range(nrows):
            vals = f.readline().split()
            row = np.zeros(ncols, dtype=np.float32)
            for c in range(min(len(vals), ncols)):
                row[c] = float(vals[c])
            z[r, :] = row
    return z, nrows, ncols, xmin, xmax, ymin, ymax

def leer_sshmax_subdom(path):
    path = Path(path)
    with path.open('rb') as f:
        if f.read(4) != b'DSBB':
            raise ValueError(f'{path} no comienza con DSBB')
        hdr = f.read(52)
        nI, nJ, loMin, loMax, laMin, laMax, t0, t1 = struct.unpack('<hh6d', hdr)
        _ = struct.unpack('<2d', f.read(16))
        data = np.fromfile(f, dtype='<f4')
        if data.size < nI*nJ:
            tmp = np.zeros(nI*nJ, dtype=np.float32)
            tmp[:data.size] = data
            data = tmp
        arr = data.reshape((nJ, nI))
    return arr, (nI, nJ), (loMin, loMax, laMin, laMax)

def colocar_subdom_en_full(full_shape, domain_bounds, sub_arr, sub_bounds):
    nrows, ncols = full_shape
    xmin, xmax, ymin, ymax = domain_bounds
    loMin, loMax, laMin, laMax = sub_bounds

    lon_full = np.linspace(xmin, xmax, ncols)
    lat_full = np.linspace(ymin, ymax, nrows)

    i0 = int(np.argmin(np.abs(lon_full - loMin)))
    i1 = int(np.argmin(np.abs(lon_full - loMax))) + 1
    j0 = int(np.argmin(np.abs(lat_full - laMin)))
    j1 = int(np.argmin(np.abs(lat_full - laMax))) + 1

    nJ, nI = sub_arr.shape
    if (j1 - j0) != nJ:
        j1 = j0 + nJ
    if (i1 - i0) != nI:
        i1 = i0 + nI

    full = np.zeros((nrows, ncols), dtype=np.float32)
    full[j0:j1, i0:i1] = sub_arr
    return full, lat_full, lon_full

def generar_netcdf(grd_file, sshmax_file, nc_file):
    bathy, nrows, ncols, xmin, xmax, ymin, ymax = leer_grilla_easywave_ascii(grd_file)
    sub_arr, (nI, nJ), sub_bounds = leer_sshmax_subdom(sshmax_file)
    max_height_full, lat, lon = colocar_subdom_en_full(
        (nrows, ncols),
        (xmin, xmax, ymin, ymax),
        sub_arr,
        sub_bounds
    )
    ds = xr.Dataset(
        data_vars={
            "original_bathy": (["lat", "lon"], bathy),
            "deformed_bathy": (["lat", "lon"], bathy),
            "max_height":     (["lat", "lon"], max_height_full),
        },
        coords={"lat": lat, "lon": lon},
        attrs={"source": "EasyWave sshmax embebido en grilla completa"}
    )
    ds.to_netcdf(nc_file)
    print(f"✅ NetCDF listo: {nc_file}")

# ================== MAIN ==================
grd_file = r"Datos/EasyWave/GrdASCII/grilla_ascii.grd"
flt_base = r"Datos/simulaciones-tsunami-hysea"
out_folder = r"Datos/EasyWave/Outputs"
os.makedirs(out_folder, exist_ok=True)

flt_files = glob.glob(os.path.join(flt_base, "**", "*.flt"), recursive=True)

for flt_file in flt_files:
    rel_path = os.path.relpath(flt_file, flt_base)
    sim_name = os.path.splitext(rel_path.replace(os.sep, "_"))[0]

    run_out = os.path.join(out_folder, sim_name)
    os.makedirs(run_out, exist_ok=True)

    print(f"Procesando {sim_name} ...")

    subprocess.run([
        "easywave",
        "-grid", os.path.abspath(grd_file),
        "-source", os.path.abspath(flt_file),
        "-time", "120"
        "-gpu"                                       #SE AÑADE PARA QUE CORRA CON LA GPU
    ], cwd=run_out)

    subprocess.run(["sshmax2png.sh", "-grd", os.path.abspath(grd_file)], cwd=run_out)
    subprocess.run(["ssh2png.sh", "-grd", os.path.abspath(grd_file), "03600"], cwd=run_out)


    flt_dir = os.path.dirname(flt_file)
    file_heig = os.path.join(run_out, "eWave.2D.png")
    file_prop = os.path.join(run_out, "eWave.2D.03600.png")
    new_heig = os.path.join(flt_dir, f"max_wave_heights.png")
    new_prop = os.path.join(flt_dir, f"wave_propagation.png")

    if os.path.exists(file_heig):
        shutil.move(file_heig, new_heig)
    if os.path.exists(file_prop):
        shutil.move(file_prop, new_prop)

    # generar NetCDF 
    sshmax_file = os.path.join(run_out, "eWave.2D.sshmax")
    nc_file = os.path.join(flt_dir, f"resultado_easywave.nc")
    if os.path.exists(sshmax_file):
        generar_netcdf(grd_file, sshmax_file, nc_file)

print("✅ Todos los procesos han finalizado.")


Procesando mw_8.1_02083_02083 ...

easyWave ver.2013-04-11
Model time = 00:00:00,   elapsed: 229446 msec
Model time = 00:10:00,   elapsed: 229826 msec
Model time = 00:20:00,   elapsed: 230750 msec
Model time = 00:30:00,   elapsed: 232769 msec
Model time = 00:40:00,   elapsed: 236007 msec
Model time = 00:50:00,   elapsed: 240292 msec
Model time = 01:00:00,   elapsed: 245243 msec
Model time = 01:10:00,   elapsed: 250950 msec
Model time = 01:20:00,   elapsed: 257665 msec
Model time = 01:30:00,   elapsed: 265220 msec
Model time = 01:40:00,   elapsed: 273378 msec
Model time = 01:50:00,   elapsed: 282219 msec
Model time = 02:00:00,   elapsed: 291724 msec
✅ NetCDF listo: Datos/simulaciones-tsunami-hysea/mw_8.1/02083/resultado_easywave.nc
Procesando mw_8.3_03529_03529 ...

easyWave ver.2013-04-11
Model time = 00:00:00,   elapsed: 230173 msec
Model time = 00:10:00,   elapsed: 230643 msec
Model time = 00:20:00,   elapsed: 231742 msec
Model time = 00:30:00,   elapsed: 233785 msec
Model time = 00: